# IDRMS Flood Risk ML — Experiment Notebook
### Barangay Kauswagan, Cagayan de Oro City
**Flood Vulnerability Profiling Using Machine Learning**

**Section:** IT3R9 | **Subject:** DS312  
**Members:** Echavia • Gorra • Guangco • Magparoc

---
This notebook covers:
1. Dataset Generation (based on IDRMS useRiskEngine.js)
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing & Encoding
4. Model Training (Decision Tree, Random Forest, Logistic Regression)
5. Model Evaluation (Accuracy, Precision, Recall, F1, Confusion Matrix, ROC-AUC)
6. Feature Importance
7. Prediction Examples
8. Saving Models

## 0. Install & Import Libraries

In [ ]:
# Install required packages (run once)
# !pip install scikit-learn imbalanced-learn pandas numpy matplotlib seaborn joblib

In [ ]:
import random
import warnings
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os

from sklearn.tree          import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble      import RandomForestClassifier
from sklearn.linear_model  import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics       import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, RocCurveDisplay
)

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)

print('Libraries loaded successfully!')

## 1. Dataset Generation

Since no public dataset contains barangay-level resident fields (zone, vulnerability tags, evacuation status), we generate a synthetic dataset using the **exact same scoring formula** from `useRiskEngine.js` in the IDRMS web app and `useRisk.js` in the mobile app.

### Zone Base Scores (from constants.js)
| Zone | Base Score | Why |
|------|-----------|-----|
| Zone 1 | 25 | Lowest flood exposure |
| Zone 2 | 42 | Moderate |
| Zone 3 | 78 | Near riverbank — HIGH flood history |
| Zone 4 | 18 | Lowest — safest zone |
| Zone 5 | 82 | Highest — closest to river |
| Zone 6 | 48 | Moderate-high |

In [ ]:
# ── Exact values from useRiskEngine.js and constants.js ──────────────────────

ZONE_BASE = {
    'Zone 1': 25, 'Zone 2': 42, 'Zone 3': 78,
    'Zone 4': 18, 'Zone 5': 82, 'Zone 6': 48,
}

VULN_WEIGHTS = {
    'Bedridden':      12,
    'PWD':            10,
    'Senior Citizen':  8,
    'Pregnant':        8,
    'Infant':          7,
}

EVAC_SCORE = {'Safe': 0, 'Evacuated': -15, 'Unaccounted': 18}
RAINY_MONTHS = set(range(6, 12))  # June to November

ZONES       = list(ZONE_BASE.keys())
VULN_TAGS   = list(VULN_WEIGHTS.keys())
EVAC_STATUS = list(EVAC_SCORE.keys())
WEATHER     = ['None', 'Medium', 'High']

print('Zone base scores:', ZONE_BASE)
print('Vulnerability weights:', VULN_WEIGHTS)
print('Evacuation score modifiers:', EVAC_SCORE)

In [ ]:
def compute_risk_score(zone, evac_status, household_members, vuln_tags,
                       rainy_season, zone_incident_count=0, weather_risk='None'):
    """
    Python re-implementation of scoreResident() from useRiskEngine.js.
    Returns a score capped between 0 and 100.
    """
    score = ZONE_BASE.get(zone, 30)
    vuln_score = sum(VULN_WEIGHTS.get(t, 5) for t in vuln_tags)
    score += min(vuln_score, 40)                         # cap vulnerability at +40
    score += EVAC_SCORE.get(evac_status, 0)
    score += min((max(int(household_members), 1) - 1) * 1.8, 12)  # each extra member +1.8, cap +12
    score += min(zone_incident_count * 6, 20)            # each incident +6, cap +20
    if weather_risk == 'High':   score += 15
    elif weather_risk == 'Medium': score += 7
    if rainy_season: score += 8
    return int(min(max(round(score), 0), 100))

def get_risk_label(score):
    if score >= 70: return 'HIGH'
    if score >= 40: return 'MEDIUM'
    return 'LOW'

# Quick test
test = compute_risk_score('Zone 5', 'Unaccounted', 6, ['Bedridden', 'Senior Citizen'], True)
print(f'Test score (Zone 5, Unaccounted, Bedridden+Senior, rainy): {test} → {get_risk_label(test)}')

In [ ]:
N = 5000

# Zone sampling weights proportional to flood exposure
zone_weights = [ZONE_BASE[z] for z in ZONES]
zone_probs   = [w / sum(zone_weights) for w in zone_weights]

records = []
for _ in range(N):
    zone      = np.random.choice(ZONES, p=zone_probs)
    evac      = random.choices(EVAC_STATUS, weights=[60, 20, 20])[0]
    n_tags    = random.choices([0,1,2,3], weights=[50,25,15,10])[0]
    tags      = random.sample(VULN_TAGS, n_tags)
    hh        = random.randint(1, 10)
    month     = random.randint(1, 12)
    rainy     = month in RAINY_MONTHS
    zone_inc  = random.choices([0,1,2,3], weights=[60,20,12,8])[0]
    weather   = random.choices(WEATHER, weights=[60,25,15])[0]

    score = compute_risk_score(zone, evac, hh, tags, rainy, zone_inc, weather)
    label = get_risk_label(score)

    records.append({
        'zone':                zone,
        'evacuation_status':   evac,
        'household_members':   hh,
        'rainy_season':        int(rainy),
        'zone_incident_count': zone_inc,
        'weather_risk':        weather,
        'tag_bedridden':       int('Bedridden'      in tags),
        'tag_pwd':             int('PWD'            in tags),
        'tag_senior_citizen':  int('Senior Citizen' in tags),
        'tag_pregnant':        int('Pregnant'       in tags),
        'tag_infant':          int('Infant'         in tags),
        'risk_score':          score,
        'risk_label':          label,
    })

df = pd.DataFrame(records)
print('Dataset shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Label distribution
print('Risk Label Distribution:')
print(df['risk_label'].value_counts())
print()
print(df['risk_label'].value_counts(normalize=True).round(3) * 100, '(percentages)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Label distribution pie chart
label_counts = df['risk_label'].value_counts()
colors = ['#e84855', '#f4a35a', '#00d68f']
axes[0].pie(label_counts, labels=label_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[0].set_title('Risk Label Distribution', fontweight='bold')

# 2. Risk label by zone
zone_risk = df.groupby(['zone', 'risk_label']).size().unstack(fill_value=0)
zone_risk.plot(kind='bar', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('Risk Label by Zone', fontweight='bold')
axes[1].set_xlabel('Zone')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Risk')

# 3. Risk score distribution
df.boxplot(column='risk_score', by='risk_label', ax=axes[2],
           boxprops=dict(color='steelblue'))
axes[2].set_title('Risk Score Distribution by Label', fontweight='bold')
axes[2].set_xlabel('Risk Label')
axes[2].set_ylabel('Score')
plt.suptitle('')

plt.tight_layout()
plt.show()

In [ ]:
# Zone base score visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Zone base scores bar chart
zone_names = list(ZONE_BASE.keys())
zone_scores = list(ZONE_BASE.values())
bar_colors = ['#e84855' if s >= 70 else '#f4a35a' if s >= 40 else '#00d68f' for s in zone_scores]
axes[0].bar(zone_names, zone_scores, color=bar_colors, edgecolor='white')
axes[0].axhline(70, color='red', linestyle='--', label='HIGH threshold (70)')
axes[0].axhline(40, color='orange', linestyle='--', label='MEDIUM threshold (40)')
axes[0].set_title('Zone Base Scores (from useRiskEngine.js)', fontweight='bold')
axes[0].set_ylabel('Base Score')
axes[0].legend()
for i, (z, s) in enumerate(zip(zone_names, zone_scores)):
    axes[0].text(i, s+1, str(s), ha='center', fontweight='bold')

# Vulnerability weights
vuln_names = list(VULN_WEIGHTS.keys())
vuln_vals  = list(VULN_WEIGHTS.values())
axes[1].barh(vuln_names, vuln_vals, color='steelblue', edgecolor='white')
axes[1].set_title('Vulnerability Tag Weights (from useRiskEngine.js)', fontweight='bold')
axes[1].set_xlabel('Score Added')
for i, v in enumerate(vuln_vals):
    axes[1].text(v+0.1, i, str(v), va='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Evacuation status vs risk label
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

evac_risk = df.groupby(['evacuation_status', 'risk_label']).size().unstack(fill_value=0)
evac_risk.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Risk Label by Evacuation Status', fontweight='bold')
axes[0].tick_params(axis='x', rotation=0)

# Vulnerability tag presence vs risk
tag_cols = ['tag_bedridden','tag_pwd','tag_senior_citizen','tag_pregnant','tag_infant']
tag_high = [df[df['risk_label']=='HIGH'][c].mean() for c in tag_cols]
tag_low  = [df[df['risk_label']=='LOW'][c].mean()  for c in tag_cols]
x = range(len(tag_cols))
axes[1].bar([i-0.2 for i in x], tag_high, 0.4, label='HIGH', color='#e84855')
axes[1].bar([i+0.2 for i in x], tag_low,  0.4, label='LOW',  color='#00d68f')
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(['Bedridden','PWD','Senior','Pregnant','Infant'], rotation=20)
axes[1].set_title('Vulnerability Tag Rate: HIGH vs LOW Risk', fontweight='bold')
axes[1].set_ylabel('Proportion with tag')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Descriptive statistics by risk label
print('Descriptive Statistics by Risk Label:')
df.groupby('risk_label')[['risk_score', 'household_members', 'zone_incident_count']].describe().round(2)

## 3. Data Preprocessing & Encoding

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())
print('\nNo missing values expected since data was generated programmatically.')

In [ ]:
# One-hot encode categorical features
# Zone: Zone 1 is reference (all zeros)
zone_dummies = pd.get_dummies(df['zone'], prefix='zone')
zone_dummies.drop(columns=['zone_Zone 1'], inplace=True, errors='ignore')

# Evacuation: Safe is reference
evac_dummies = pd.get_dummies(df['evacuation_status'], prefix='evac')
evac_dummies.drop(columns=['evac_Safe'], inplace=True, errors='ignore')

# Weather: None is reference
wx_dummies = pd.get_dummies(df['weather_risk'], prefix='weather')
wx_dummies.drop(columns=['weather_None'], inplace=True, errors='ignore')

tag_cols = ['tag_bedridden','tag_pwd','tag_senior_citizen','tag_pregnant','tag_infant']
num_cols = ['household_members','rainy_season','zone_incident_count','risk_score']

X = pd.concat([zone_dummies, evac_dummies, wx_dummies, df[tag_cols], df[num_cols]], axis=1)
y = df['risk_label']

print('Feature matrix shape:', X.shape)
print('Features:', list(X.columns))
X.head()

In [ ]:
# Correlation heatmap (numerical features only)
num_feats = num_cols + tag_cols
plt.figure(figsize=(10, 7))
corr = df[num_feats + ['risk_label']].copy()
# Encode risk_label for correlation
corr['risk_label_num'] = corr['risk_label'].map({'LOW':0,'MEDIUM':1,'HIGH':2})
corr = corr.drop(columns=['risk_label'])
sns.heatmap(corr.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 80/20 stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {len(X_train)} rows')
print(f'Test size:  {len(X_test)} rows')
print('\nClass distribution in training set:')
print(y_train.value_counts())
print('\nClass distribution in test set:')
print(y_test.value_counts())

In [ ]:
# Scale for Logistic Regression
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
print('Scaling done. StandardScaler fitted on training set only (no data leakage).')

## 4. Model Training

We train 3 models as specified in the IDRMS ML paper:
- **Decision Tree** — interpretable, entropy criterion
- **Random Forest** — ensemble of 100 trees, most accurate
- **Logistic Regression** — linear baseline

In [ ]:
# ── Decision Tree ──────────────────────────────────────────────────────────
dt = DecisionTreeClassifier(
    criterion='entropy', max_depth=8,
    min_samples_leaf=5, random_state=42
)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
print('Decision Tree trained!')
print(f'Accuracy: {accuracy_score(y_test, y_pred_dt):.2%}')

In [ ]:
# ── Random Forest ──────────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=100, max_depth=8, max_features='sqrt',
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print('Random Forest trained!')
print(f'Accuracy: {accuracy_score(y_test, y_pred_rf):.2%}')

In [ ]:
# ── Logistic Regression ────────────────────────────────────────────────────
lr = LogisticRegression(
    max_iter=1000, class_weight='balanced',
    solver='lbfgs', random_state=42
)
lr.fit(X_train_s, y_train)
y_pred_lr = lr.predict(X_test_s)
print('Logistic Regression trained!')
print(f'Accuracy: {accuracy_score(y_test, y_pred_lr):.2%}')

## 5. Model Evaluation

In [ ]:
models_eval = {
    'Decision Tree':       (y_pred_dt, dt),
    'Random Forest':       (y_pred_rf, rf),
    'Logistic Regression': (y_pred_lr, lr),
}

print('='*65)
print(f'{"Model":<22} {"Accuracy":>10} {"P(HIGH)":>10} {"R(HIGH)":>10} {"F1(HIGH)":>10}')
print('='*65)
for name, (y_pred, _) in models_eval.items():
    acc = accuracy_score(y_test, y_pred)
    p   = precision_score(y_test, y_pred, labels=['HIGH'], average='macro', zero_division=0)
    r   = recall_score   (y_test, y_pred, labels=['HIGH'], average='macro', zero_division=0)
    f1  = f1_score       (y_test, y_pred, labels=['HIGH'], average='macro', zero_division=0)
    print(f'{name:<22} {acc:>10.2%} {p:>10.2%} {r:>10.2%} {f1:>10.2%}')
print('='*65)

In [ ]:
# Full classification report for each model
for name, (y_pred, _) in models_eval.items():
    print(f'\n── {name} ──')
    print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrices for all 3 models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
labels = ['HIGH', 'MEDIUM', 'LOW']

for ax, (name, (y_pred, _)) in zip(axes, models_eval.items()):
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    disp = ConfusionMatrixDisplay(cm, display_labels=labels)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\nAccuracy: {accuracy_score(y_test, y_pred):.2%}', fontweight='bold')

plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Cross-validation scores (5-fold)
print('5-Fold Cross-Validation Accuracy:')
for name, model, X_cv, y_cv in [
    ('Decision Tree',       dt, X_train.values, y_train),
    ('Random Forest',       rf, X_train.values, y_train),
    ('Logistic Regression', lr, X_train_s,       y_train),
]:
    scores = cross_val_score(model, X_cv, y_cv, cv=5, scoring='accuracy')
    print(f'  {name:<22}: {scores.mean():.4f} ± {scores.std():.4f}')

## 6. Feature Importance

In [ ]:
# Random Forest Feature Importances
fi = pd.DataFrame({
    'feature':    list(X.columns),
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 7))
colors_fi = ['#e84855' if i < 3 else 'steelblue' for i in range(len(fi))]
plt.barh(fi['feature'][::-1], fi['importance'][::-1], color=colors_fi[::-1])
plt.title('Feature Importances — Random Forest', fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('\nTop 10 Features:')
print(fi.head(10).to_string(index=False))

In [ ]:
# Decision Tree Visualization (first 3 levels)
plt.figure(figsize=(20, 8))
plot_tree(dt, feature_names=list(X.columns),
          class_names=dt.classes_,
          filled=True, max_depth=3,
          fontsize=9, rounded=True)
plt.title('Decision Tree — First 3 Levels', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Print readable decision tree rules (first 10 lines)
rules = export_text(dt, feature_names=list(X.columns))
print('Decision Tree Rules (first 50 lines):')
print('\n'.join(rules.split('\n')[:50]))

## 7. Prediction Examples

These are the same examples you would send to `POST /api/predict/resident/` in the FastAPI Swagger UI.

In [ ]:
def predict_example(zone, evac_status, household_members, vuln_tags,
                    rainy_season=True, zone_incident_count=0,
                    weather_risk='None', model=rf, model_name='Random Forest'):
    """Quick prediction function for demo purposes."""
    risk_score = compute_risk_score(zone, evac_status, household_members,
                                    vuln_tags, rainy_season, zone_incident_count, weather_risk)
    # Build feature row
    row = {col: 0 for col in X.columns}
    if f'zone_{zone}' in row: row[f'zone_{zone}'] = 1
    if evac_status == 'Evacuated'   and 'evac_Evacuated'   in row: row['evac_Evacuated']   = 1
    if evac_status == 'Unaccounted' and 'evac_Unaccounted' in row: row['evac_Unaccounted'] = 1
    if weather_risk == 'Medium' and 'weather_Medium' in row: row['weather_Medium'] = 1
    if weather_risk == 'High'   and 'weather_High'   in row: row['weather_High']   = 1
    tag_map = {'Bedridden':'tag_bedridden','PWD':'tag_pwd',
               'Senior Citizen':'tag_senior_citizen','Pregnant':'tag_pregnant','Infant':'tag_infant'}
    for tag, col in tag_map.items():
        if tag in vuln_tags and col in row: row[col] = 1
    row['household_members']   = household_members
    row['rainy_season']        = int(rainy_season)
    row['zone_incident_count'] = zone_incident_count
    row['risk_score']          = risk_score
    X_pred = pd.DataFrame([row])[X.columns]
    label = model.predict(X_pred)[0]
    proba = model.predict_proba(X_pred)[0]
    conf  = round(max(proba)*100, 1)
    print(f'  Risk Score : {risk_score}')
    print(f'  Prediction : {label}  ({model_name}, confidence: {conf}%)')
    return label

print('='*55)
print('Example 1 — HIGH Risk (Zone 5, Bedridden, Unaccounted)')
print('='*55)
predict_example('Zone 5','Unaccounted', 6, ['Bedridden','Senior Citizen'])

print('\n' + '='*55)
print('Example 2 — MEDIUM Risk (Zone 4, Pregnant, Safe)')
print('='*55)
predict_example('Zone 4','Safe', 4, ['Pregnant'])

print('\n' + '='*55)
print('Example 3 — LOW Risk (Zone 1, no tags, Evacuated)')
print('='*55)
predict_example('Zone 1','Evacuated', 2, [], rainy_season=False)

print('\n' + '='*55)
print('Example 4 — HIGH Risk (Zone 3, PWD + Infant, Unaccounted)')
print('='*55)
predict_example('Zone 3','Unaccounted', 8, ['PWD','Infant'])

print('\n' + '='*55)
print('Example 5 — HIGH Risk with bad weather')
print('='*55)
predict_example('Zone 2','Unaccounted', 5, ['Pregnant'], weather_risk='High', zone_incident_count=2)

## 8. Save Models

In [ ]:
os.makedirs('ml/model', exist_ok=True)

joblib.dump(dt,     'ml/model/decision_tree.pkl')
joblib.dump(rf,     'ml/model/random_forest.pkl')
joblib.dump(lr,     'ml/model/logistic_regression.pkl')
joblib.dump(scaler, 'ml/model/scaler.pkl')

with open('ml/model/feature_columns.json', 'w') as f:
    json.dump(list(X.columns), f, indent=2)

df.to_csv('ml/model/training_data.csv', index=False)
fi.to_csv('ml/model/feature_importances.csv', index=False)

summary = {
    'trained_at':   datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'dataset_size': N,
    'features':     list(X.columns),
    'models': {
        'Decision Tree':       {'accuracy': round(accuracy_score(y_test,y_pred_dt),4),
                                'f1_high':  round(f1_score(y_test,y_pred_dt,labels=['HIGH'],average='macro'),4)},
        'Random Forest':       {'accuracy': round(accuracy_score(y_test,y_pred_rf),4),
                                'f1_high':  round(f1_score(y_test,y_pred_rf,labels=['HIGH'],average='macro'),4)},
        'Logistic Regression': {'accuracy': round(accuracy_score(y_test,y_pred_lr),4),
                                'f1_high':  round(f1_score(y_test,y_pred_lr,labels=['HIGH'],average='macro'),4)},
    }
}
with open('ml/model/model_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

with open('ml/model/decision_tree_rules.txt', 'w') as f:
    f.write(export_text(dt, feature_names=list(X.columns)))

print('All model files saved to ml/model/')
print(os.listdir('ml/model/'))

## Summary

| Model | Accuracy | F1 (HIGH) | Best For |
|---|---|---|---|
| Decision Tree | ~100% | ~100% | Explaining decisions to officials |
| Random Forest | ~100% | ~100% | Actual predictions in the system |
| Logistic Regression | ~98% | ~99% | Baseline comparison |

The most important features are:
1. `risk_score` — computed by the IDRMS rule engine
2. `zone_Zone 5` — highest flood-prone zone
3. `zone_Zone 3` — second highest flood-prone zone
4. `evac_Evacuated` — already evacuated reduces risk
5. `evac_Unaccounted` — not confirmed safe increases risk

The ML model validates and confirms the IDRMS rule engine classifications, providing confidence scores and the ability to rank all residents automatically when a flood warning is triggered.